In [1]:
!pip install openpyxl
!pip install gurobipy
import openpyxl
import gurobipy as gp
from gurobipy import GRB

# 1. Data input

In [2]:
# Input/Output
input_workbook_file = openpyxl.load_workbook("input.xlsx")
output_workbook_file_name = "output.xlsx"

# Set of zones
zones = ["Cargo", "Pax", "Vehicles", "Train"]

# Set of shifts
shifts = ["M", "A", "N"]

# The sequence of zones for the next month assignment
zone_sequence = {
    "Cargo":"Pax",
    "Pax":"Vehicles",
    "Vehicles":"Train",
    "Train":"Cargo"
    }

# 2. Question 1: Staff-to-zone assignment

## 2.1. Supporting functions to get the required data

In [3]:
# Scan all rows in the first column of the worksheet to determine which row contains the given table name
def get_row_contain_table_name(worksheet, table_name):
    for row in range(1, worksheet.max_row + 1):
        value = worksheet.cell(row, 1).value
        if isinstance(value, str) and value.strip() == table_name:
            return row
    return None

# Scan through columns to find workday labels (Day 1, Day 2, ...) until we hit an empty cell to determine the end of the days
def get_workday(worksheet, header_row):
    workday_cols = []  # List to hold the column numbers and day labels
    start_col = 3  # Start from the third column (column C in the worksheet) for days

    while True:
        value = worksheet.cell(header_row, start_col).value
        if value is None:
            break
        workday_cols.append((start_col, str(value).strip()))
        start_col += 1

    return workday_cols

# For "Schedule Table" table, return the columns that contain day labels and the rows that contain staff names
def parse_schedule_table(worksheet, table_name="Schedule Table"):
    table_name_row = get_row_contain_table_name(worksheet, table_name)  # Find the row number of the table name
    header_row = table_name_row + 1  # The header row is immediately after the table name row
    start_row = header_row + 1  # The first row of staff data starts after the header row

    # Get the workday columns
    workday_cols = get_workday(worksheet, header_row)

    # Get the staff name rows
    # Loop through rows to find staff names until hitting a non-staff name value/row
    staff_rows = []  # List to hold the row indexes and staff names
    staff_row = start_row  # Start from the first row of staff data

    while True:
        staff_name = worksheet.cell(staff_row, 2).value  # Staff names are in the second column (column B in the worksheet)
        if not isinstance(staff_name, str) or not staff_name.startswith("Staff "):
            break
        staff_rows.append((staff_row, staff_name))
        staff_row += 1

    # Return the list of day columns and staff rows for further processing
    return workday_cols, staff_rows

# For "Demand Table" table, return the demand for each zone and shift
def parse_demand(worksheet, table_name="Demand Table"):
    table_name_row = get_row_contain_table_name(worksheet, table_name)
    header_row = table_name_row + 1
    start_row = header_row + 1

    # Get the workday columns
    workday_cols = get_workday(worksheet, header_row)

    # Build the demand dictionary
    # demand[(zone, shift, workday)] = demand_value
    demand = {}
    demand_row = start_row  # Start from the first row of demand data

    while True:
        zone = worksheet.cell(demand_row, 1).value  # Zone names are in the first column (column A in the worksheet)
        shift = worksheet.cell(demand_row, 2).value  # Shift names are in the second column (column B in the worksheet)

        # Stop when reaching the end of the demand table
        if zone is None and shift is None:
            break

        if zone not in zones or shift not in shifts:
            demand_row += 1
            continue

        # For each zone-shift row, loop through each workday column to get the demand value of that workday
        for workday_col, workday in workday_cols:
            demand_value = worksheet.cell(demand_row, workday_col).value
            demand[(zone, shift, workday)] = int(demand_value) if demand_value is not None else 0

        demand_row += 1

    return [workday for _, workday in workday_cols], demand

# Build the availability parameter for assignment from the schedule table
# availability_to_be_assigned[(staff_name, shift, workday)] = 1 if the staff works that shift on that workday, else 0
def build_availability_to_be_assigned(worksheet, workday_cols, staff_rows):
    availability_to_be_assigned = {}

    for staff_row, staff_name in staff_rows:
        for workday_col, workday in workday_cols:
            shift_value = worksheet.cell(staff_row, workday_col).value
            shift_value = str(shift_value).strip() if shift_value is not None else ""

            for shift in shifts:
                availability_to_be_assigned[(staff_name, shift, workday)] = 1 if shift_value == shift else 0

    return availability_to_be_assigned


## 2.2. Optimization model

In [4]:
def staff_to_zone_assignment(worksheet, demand):
    # Parse the schedule table
    workday_cols, staff_rows = parse_schedule_table(worksheet)

    # Build staff list and availability_to_be_assigned parameter
    staff_names = [staff_name for staff_index, staff_name in staff_rows]
    workdays = [workday for workday_index, workday in workday_cols]
    availability_to_be_assigned = build_availability_to_be_assigned(
        worksheet,
        workday_cols,
        staff_rows
    )

    # Create model
    model = gp.Model("staff_to_zone_assignment")

    # Decision variables
    x = model.addVars(staff_names, zones, vtype=GRB.BINARY, name="assign a staff to a zone")

    # Balancing variables to try to keep the staff assignment between zone relatively even
    max_zone_load = model.addVar(vtype=GRB.INTEGER, name="max_zone_load")
    min_zone_load = model.addVar(vtype=GRB.INTEGER, name="min_zone_load")

    # As we are not required to optimize anything, all is about finding a feasible solution for the problem
    # Therefore we can set "model.setObjective(0)"
    # However, we can set a supporting objective of "minimizing the difference between the largest and smallest zone loads"
    # or "balance the load between zones" to have a fairer assignment
    model.setObjective(max_zone_load - min_zone_load, GRB.MINIMIZE)

    # Constraint 1: Each staff must be assigned to exactly one zone (for the whole month)
    for staff_name in staff_names:
        model.addConstr(
            gp.quicksum(x[staff_name, zone] for zone in zones) == 1,
            name=f"one_zone_per_staff_{staff_name}"
        )

    # Constraint 2: Satisfy the demand for every zone-shift-workday combination
    for zone in zones:
        for shift in shifts:
            for workday in workdays:
                required_demand = demand.get((zone, shift, workday), 0)
                model.addConstr(
                    gp.quicksum(
                        availability_to_be_assigned[(staff_name, shift, workday)] * x[staff_name, zone]
                        for staff_name in staff_names
                    ) >= required_demand,
                    name=f"demand_{zone}_{shift}_{workday}"
                )

    # Constraint 3: Define max and min zone loads for balancing purpose
    for zone in zones:
        zone_load = gp.quicksum(x[staff_name, zone] for staff_name in staff_names)
        model.addConstr(zone_load <= max_zone_load, name=f"max_load_{zone}")
        model.addConstr(zone_load >= min_zone_load, name=f"min_load_{zone}")

    # Solve the model
    model.optimize()

    if model.Status != GRB.OPTIMAL:
        raise Exception(f"Model is not solved optimally. Solver status: {model.Status}")

    # Extract the assignment result
    staff_to_zone_assignment_result = {}
    for staff_name in staff_names:
        for zone in zones:
            if x[staff_name, zone].X > 0.5:
                staff_to_zone_assignment_result[staff_name] = zone
                break

    return staff_to_zone_assignment_result

# 3. Question 2: Staff-to-zone assignment for next month

## 3.1. Supporting functions to get the required data

In [5]:
# Read the previous month zone assignment from "Previous Month Schedule Table"
def parse_previous_month_zone_assignment(worksheet, table_name="Previous Month Schedule Table"):
    table_name_row = get_row_contain_table_name(worksheet, table_name)
    header_row = table_name_row + 1
    start_row = header_row + 1

    previous_month_zone_assignment = {}
    staff_row = start_row

    while True:
        staff_name = worksheet.cell(staff_row, 2).value
        if not isinstance(staff_name, str) or not staff_name.startswith("Staff "):
            break

        assigned_zone = None

        # Scan across the row to find the first working cell containing "Shift Zone"
        for col in range(3, worksheet.max_column + 1):
            cell_value = worksheet.cell(staff_row, col).value
            if isinstance(cell_value, str):
                cell_value = cell_value.strip()
                if cell_value != "O" and " " in cell_value:
                    shift_part, zone_part = cell_value.split(" ", 1)
                    if zone_part in zones:
                        assigned_zone = zone_part
                        break

        previous_month_zone_assignment[staff_name] = assigned_zone
        staff_row += 1

    return previous_month_zone_assignment

## 3.2. Rule-based assignment

In [6]:
# Rule-based staff-to-zone assignment for next month
def staff_to_zone_assignment_next_month(worksheet):
    workday_cols, staff_rows = parse_schedule_table(worksheet)
    previous_month_zone_assignment = parse_previous_month_zone_assignment(worksheet)

    next_month_assignment = {}

    for _, staff_name in staff_rows:
        previous_zone = previous_month_zone_assignment.get(staff_name)

        if previous_zone in zone_sequence:
            next_month_assignment[staff_name] = zone_sequence[previous_zone]
        else:
            # Fallback in case the previous month zone is missing
            next_month_assignment[staff_name] = "Cargo"

    return next_month_assignment

# 4. Write the assignment resuls back to the workbook

In [7]:
# Fill the "Schedule Table" table cells with the format "Shift Zone"
# Keep "O" unchanged
def fill_zone_assignment_into_schedule(worksheet, assignment):
    workday_cols, staff_rows = parse_schedule_table(worksheet)

    for staff_row, staff_name in staff_rows:
        assigned_zone = assignment[staff_name]

        for workday_col, workday in workday_cols:
            shift_value = worksheet.cell(staff_row, workday_col).value

            if isinstance(shift_value, str):
                shift_value = shift_value.strip()

                if shift_value in shifts:
                    worksheet.cell(staff_row, workday_col).value = f"{shift_value} {assigned_zone}"
                elif shift_value == "O":
                    worksheet.cell(staff_row, workday_col).value = "O"

# 5. Run all

In [8]:
# Main flow
description_worksheet = input_workbook_file["Description"]
q1_answer_worksheet = input_workbook_file["Q1 Answer"]
q2_answer_worksheet = input_workbook_file["Q2 Answer"]

# Parse demand from Description sheet
workdays, demand = parse_demand(description_worksheet)

# Solve Question 1 and write the assignment back to the sheet
q1_assignment = staff_to_zone_assignment(q1_answer_worksheet, demand)
fill_zone_assignment_into_schedule(q1_answer_worksheet, q1_assignment)

# Build Q2 assignment by using the previous month zone rotation rule
q2_assignment = staff_to_zone_assignment_next_month(q2_answer_worksheet)
fill_zone_assignment_into_schedule(q2_answer_worksheet, q2_assignment)

# Export the output
input_workbook_file.save(output_workbook_file_name)
print(f"Done.")

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (linux64 - "Ubuntu 22.04.5 LTS")

CPU model: Intel(R) Xeon(R) CPU @ 2.20GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 1 physical cores, 2 logical processors, using up to 2 threads

Optimize a model with 399 rows, 78 columns and 2000 nonzeros (Min)
Model fingerprint: 0x0d444e2a
Model has 2 linear objective coefficients
Variable types: 0 continuous, 78 integer (76 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]

Found heuristic solution: objective 2.0000000
Presolve removed 356 rows and 0 columns
Presolve time: 0.00s
Presolved: 43 rows, 78 columns, 312 nonzeros
Variable types: 0 continuous, 78 integer (76 binary)

Root relaxation: objective 0.000000e+00, 46 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objectiv